## Reading the column names and types

In [0]:
df = spark.table("pfin_dev.bronze.elections_canada_contributions_raw")
df.printSchema()
df.show(5, truncate=False)

root
 |-- political_entity: string (nullable = true)
 |-- recipient_id: string (nullable = true)
 |-- recipient: string (nullable = true)
 |-- recipient_last_name: string (nullable = true)
 |-- recipient_first_name: string (nullable = true)
 |-- recipient_middle_initial: string (nullable = true)
 |-- political_party_of_recipient: string (nullable = true)
 |-- electoral_district: string (nullable = true)
 |-- electoral_event: string (nullable = true)
 |-- fiscal_election_date: string (nullable = true)
 |-- form_id: string (nullable = true)
 |-- financial_report: string (nullable = true)
 |-- part_number_of_return: string (nullable = true)
 |-- financial_report_part: string (nullable = true)
 |-- contributor_type: string (nullable = true)
 |-- contributor_name: string (nullable = true)
 |-- contributor_last_name: string (nullable = true)
 |-- contributor_first_name: string (nullable = true)
 |-- contributor_middle_initial: string (nullable = true)
 |-- contributor_city: string (nullable 

## Apply data types to each column

In [0]:
from pyspark.sql import functions as F

df = spark.table("pfin_dev.bronze.elections_canada_contributions_raw")

# ── 1. Distinct values in categorical columns ──
print("=== CARDINALITY ===")
for col in ["political_entity", "electoral_event", "contributor_type", 
            "contributor_province", "political_party_of_recipient",
            "contribution_given_through", "financial_report_part"]:
    distinct = df.select(col).distinct().collect()
    vals = [r[0] for r in distinct]
    print(f"\n{col} ({len(vals)} distinct):")
    for v in sorted(vals, key=lambda x: str(x)):
        print(f"  - {v}")

# ── 2. NULL counts for every column ──
print("\n=== NULL COUNTS ===")
null_counts = df.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in df.columns if c not in ("_source_file", "_ingested_at")
]).collect()[0]
for c in df.columns:
    if c not in ("_source_file", "_ingested_at"):
        print(f"  {c}: {null_counts[c]}")

# ── 3. Duplicate check on full row (excluding metadata) ──
src_cols = [c for c in df.columns if c not in ("_source_file", "_ingested_at")]
total = df.count()
distinct_rows = df.select(src_cols).distinct().count()
print(f"\n=== DUPLICATES ===")
print(f"  Total rows:    {total}")
print(f"  Distinct rows: {distinct_rows}")
print(f"  Exact dupes:   {total - distinct_rows}")

# ── 4. Recipient ID cardinality ──
print(f"\n=== RECIPIENTS ===")
print(f"  Distinct recipient_id: {df.select('recipient_id').distinct().count()}")
print(f"  Distinct recipient (name): {df.select('recipient').distinct().count()}")

# ── 5. Date format check ──
print(f"\n=== DATE SAMPLES ===")
df.select("fiscal_election_date").distinct().sort("fiscal_election_date").show(20, truncate=False)
df.select("contribution_received_date").distinct().sort("contribution_received_date").show(20, truncate=False)

# ── 6. Monetary amount samples (check for non-numeric) ──
print(f"\n=== MONETARY EDGE CASES ===")
df.filter(~F.col("monetary_amount").rlike(r"^\s*-?\d+\.?\d*\s*$")).select("monetary_amount").distinct().show(20, truncate=False)
df.filter(~F.col("non_monetary_amount").rlike(r"^\s*-?\d+\.?\d*\s*$")).select("non_monetary_amount").distinct().show(20, truncate=False)

=== CARDINALITY ===

political_entity (7 distinct):
  - Candidates
  - Leadership contestants
  - Nomination contestants
  - Registered associations
  - Registered parties
  - Ã¯Â»Â¿Leadership contestants
  - Ã¯Â»Â¿Registered parties

electoral_event (6 distinct):
  - 45th general election
  - Annual
  - April 14th, 2025, By-election
  - August 18, 2025, By-election
  - None
  - Quarterly

contributor_type (1 distinct):
  - Individuals

contributor_province (36 distinct):
  - 7
  - AB
  - AB 
  - Alberta
  - Alberta 
  - BC
  - BC 
  - British Columbia
  - M4B 1N7
  - M6G 2T1
  - MB
  - MB 
  - MI 
  - NB
  - NL
  - NS
  - NS 
  - NT
  - NU
  - None
  - ON
  - ON 
  - ONT
  - ONT.
  - On
  - Ontario
  - PE
  - PEI 
  - QC
  - QC 
  - QuÃÂ©bec
  - SK
  - SK 
  - Saskatchewan
  - YT
  - aQC

political_party_of_recipient (16 distinct):
  - Bloc QuÃÂ©bÃÂ©cois
  - Canadian Future Party
  - Centrist Party of Canada
  - Christian Heritage Party of Canada
  - Communist Party of Canada
  - C

## All Phases

In [0]:
# ============================================================
# PFIN | Phase 2 | Silver Transformation
# Notebook:  02_transform_silver
# Source:    pfin_dev.bronze.elections_canada_contributions_raw
# Target:    pfin_dev.silver.{recipients, contributors,
#            electoral_events, contributions}
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DecimalType, IntegerType
from datetime import datetime

# ── CONFIG ──────────────────────────────────────────────────
TARGET_CATALOG  = "pfin_dev"
TARGET_SCHEMA   = "silver"
SOURCE_TABLE    = f"{TARGET_CATALOG}.bronze.elections_canada_contributions_raw"

TABLES = {
    "recipients":       f"{TARGET_CATALOG}.{TARGET_SCHEMA}.recipients",
    "contributors":     f"{TARGET_CATALOG}.{TARGET_SCHEMA}.contributors",
    "electoral_events": f"{TARGET_CATALOG}.{TARGET_SCHEMA}.electoral_events",
    "contributions":    f"{TARGET_CATALOG}.{TARGET_SCHEMA}.contributions",
}

# ============================================================
# STEP 1: READ BRONZE + GLOBAL CLEANING
# ============================================================
print(f"[{datetime.now()}] Reading Bronze...")
df_raw = spark.table(SOURCE_TABLE)
total_raw = df_raw.count()
print(f"  Bronze rows: {total_raw:,}")

# ── Dedup exact duplicates (exclude metadata columns) ───────
src_cols = [c for c in df_raw.columns if c not in ("_source_file", "_ingested_at")]
df = df_raw.dropDuplicates(src_cols)
deduped = df.count()
print(f"  After dedup:  {deduped:,}  (dropped {total_raw - deduped:,})")

# ── Strip BOM from political_entity ─────────────────────────
df = df.withColumn(
    "political_entity",
    F.regexp_replace("political_entity", "^Ã¯Â»Â¿", "")
)

# ── Fix double-encoded UTF-8 on text columns ────────────────
# Known patterns: ÃÂ© → é, ÃÂ¨ → è, ÃÂ‰ → É
encoding_fixes = [
    ("Ã©", "é"), ("Ã¨", "è"), ("Ã‰", "É"),
    ("ÃÂ©", "é"), ("ÃÂ¨", "è"), ("ÃÂ‰", "É"),
]
for col_name in ["political_party_of_recipient", "electoral_district",
                 "recipient", "recipient_last_name", "electoral_event"]:
    for bad, good in encoding_fixes:
        df = df.withColumn(col_name, F.regexp_replace(F.col(col_name), bad, good))

# ── Standardize province (native when/otherwise) ────────────
prov_trimmed = F.upper(F.trim(F.col("contributor_province")))
# Remove trailing dots
prov_clean = F.regexp_replace(prov_trimmed, r"\.$", "")

df = df.withColumn("contributor_province",
    F.when(prov_clean.isin("AB", "ALBERTA"), "AB")
     .when(prov_clean.isin("BC", "BRITISH COLUMBIA"), "BC")
     .when(prov_clean.isin("MB"), "MB")
     .when(prov_clean.isin("NB"), "NB")
     .when(prov_clean.isin("NL"), "NL")
     .when(prov_clean.isin("NS"), "NS")
     .when(prov_clean.isin("NT"), "NT")
     .when(prov_clean.isin("NU"), "NU")
     .when(prov_clean.isin("ON", "ONT", "ONTARIO"), "ON")
     .when(prov_clean.isin("PE", "PEI"), "PE")
     .when(prov_clean.rlike("(?i)(^QC$|^AQC$|QU|BEC)"), "QC")
     .when(prov_clean.isin("SK", "SASKATCHEWAN"), "SK")
     .when(prov_clean.isin("YT"), "YT")
     .otherwise(None)  # garbage → NULL
)

# ── Clean postal code: uppercase, strip spaces ──────────────
df = df.withColumn(
    "contributor_postal_code",
    F.upper(F.regexp_replace(F.trim(F.col("contributor_postal_code")), r"\s+", ""))
)

# ── Trim + uppercase city ───────────────────────────────────
df = df.withColumn("contributor_city", F.upper(F.trim(F.col("contributor_city"))))

# ── Nullify "None" strings ──────────────────────────────────
for col_name in ["electoral_event", "contribution_given_through", "leadership_contestant"]:
    df = df.withColumn(
        col_name,
        F.when(F.trim(F.col(col_name)).isin("None", ""), None).otherwise(F.col(col_name))
    )

# ── Fix "0025-*" → "2025-*" date typo ──────────────────────
df = df.withColumn(
    "contribution_received_date",
    F.regexp_replace("contribution_received_date", "^0025-", "2025-")
)

# ── Cast dates ──────────────────────────────────────────────
df = df.withColumn("fiscal_election_date", F.col("fiscal_election_date").cast(DateType()))
df = df.withColumn("contribution_received_date", F.col("contribution_received_date").cast(DateType()))

# ── Cast monetary amounts ───────────────────────────────────
df = df.withColumn("monetary_amount", F.trim(F.col("monetary_amount")).cast(DecimalType(12, 2)))
df = df.withColumn("non_monetary_amount", F.trim(F.col("non_monetary_amount")).cast(DecimalType(12, 2)))
df = df.withColumn("total_amount",
    F.coalesce(F.col("monetary_amount"), F.lit(0).cast(DecimalType(12, 2))) +
    F.coalesce(F.col("non_monetary_amount"), F.lit(0).cast(DecimalType(12, 2)))
)

# ── Cast recipient_id to INT ────────────────────────────────
df = df.withColumn("recipient_id", F.col("recipient_id").cast(IntegerType()))

# ── Derive fiscal_year ──────────────────────────────────────
df = df.withColumn("fiscal_year", F.year("fiscal_election_date"))

# ── Generate keys (native SHA-256) ──────────────────────────
df = df.withColumn("contributor_key",
    F.sha2(F.concat_ws("|",
        F.upper(F.trim(F.coalesce(F.col("contributor_last_name"), F.lit("")))),
        F.upper(F.trim(F.coalesce(F.col("contributor_first_name"), F.lit("")))),
        F.upper(F.trim(F.coalesce(F.col("contributor_postal_code"), F.lit(""))))
    ), 256)
)

df = df.withColumn("electoral_event_key",
    F.sha2(F.concat_ws("|",
        F.upper(F.trim(F.coalesce(F.col("electoral_event"), F.lit("")))),
        F.coalesce(F.col("fiscal_election_date").cast("string"), F.lit(""))
    ), 256)
)

# ── Add transformation timestamp ────────────────────────────
df = df.withColumn("_transformed_at", F.current_timestamp())

# 
print(f"[{datetime.now()}] Global cleaning complete.")

# ============================================================
# STEP 2: WRITE silver.recipients
# ============================================================
print(f"[{datetime.now()}] Writing recipients...")

df_recipients = (
    df.select(
        "recipient_id",
        F.col("recipient").alias("recipient_name"),
        "recipient_last_name",
        "recipient_first_name",
        "recipient_middle_initial",
        "political_entity",
        F.col("political_party_of_recipient").alias("political_party"),
        "electoral_district",
    )
    .dropDuplicates(["recipient_id"])
)

(df_recipients.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLES["recipients"]))

rc = spark.table(TABLES["recipients"]).count()
print(f"  recipients: {rc:,} rows")

# ============================================================
# STEP 3: WRITE silver.contributors
# ============================================================
print(f"[{datetime.now()}] Writing contributors...")

df_contributors = (
    df.select(
        "contributor_key",
        "contributor_type",
        "contributor_name",
        "contributor_last_name",
        "contributor_first_name",
        "contributor_middle_initial",
        "contributor_city",
        "contributor_province",
        "contributor_postal_code",
    )
    .dropDuplicates(["contributor_key"])
)

(df_contributors.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLES["contributors"]))

rc = spark.table(TABLES["contributors"]).count()
print(f"  contributors: {rc:,} rows")

# ============================================================
# STEP 4: WRITE silver.electoral_events
# ============================================================
print(f"[{datetime.now()}] Writing electoral_events...")

df_events = (
    df.select(
        "electoral_event_key",
        "electoral_event",
        "fiscal_election_date",
        "fiscal_year",
    )
    .dropDuplicates(["electoral_event_key"])
)

(df_events.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLES["electoral_events"]))

rc = spark.table(TABLES["electoral_events"]).count()
print(f"  electoral_events: {rc:,} rows")

# ============================================================
# STEP 5: WRITE silver.contributions
# ============================================================
print(f"[{datetime.now()}] Writing contributions...")

df_contributions = (
    df.select(
        F.monotonically_increasing_id().alias("contribution_id"),
        "recipient_id",
        "contributor_key",
        "electoral_event_key",
        "form_id",
        "financial_report",
        "part_number_of_return",
        "financial_report_part",
        "contribution_received_date",
        "monetary_amount",
        "non_monetary_amount",
        "total_amount",
        "leadership_contestant",
        "_source_file",
        "_ingested_at",
        "_transformed_at",
    )
)

(df_contributions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLES["contributions"]))

rc = spark.table(TABLES["contributions"]).count()
print(f"  contributions: {rc:,} rows")

# ============================================================
# STEP 6: ENABLE LIQUID CLUSTERING + PREDICTIVE OPTIMIZATION
# ============================================================
print(f"[{datetime.now()}] Configuring tables...")

spark.sql(f"ALTER TABLE {TABLES['contributions']} CLUSTER BY (contribution_received_date)")

for name, table in TABLES.items():
    spark.sql(f"ALTER TABLE {table} ENABLE PREDICTIVE OPTIMIZATION")
    print(f"  {name}: predictive optimization enabled")

# ── Unpersist cache ─────────────────────────────────────────
## df.unpersist()

# ============================================================
# STEP 7: VALIDATION SUMMARY
# ============================================================
print(f"\n{'='*60}")
print(f"PHASE 2 COMPLETE — Silver Layer Summary")
print(f"{'='*60}")
for name, table in TABLES.items():
    count = spark.table(table).count()
    print(f"  {table}: {count:,} rows")
print(f"  Timestamp: {datetime.now()}")
print(f"{'='*60}")

[2026-06-01 02:35:25.321958] Reading Bronze...
  Bronze rows: 284,136
  After dedup:  282,098  (dropped 2,038)
[2026-06-01 02:35:27.706679] Global cleaning complete.
[2026-06-01 02:35:27.706740] Writing recipients...
  recipients: 1,004 rows
[2026-06-01 02:35:29.862805] Writing contributors...
  contributors: 123,453 rows
[2026-06-01 02:35:32.914394] Writing electoral_events...
  electoral_events: 44 rows
[2026-06-01 02:35:35.386435] Writing contributions...
  contributions: 282,098 rows
[2026-06-01 02:35:39.014879] Configuring tables...
  recipients: predictive optimization enabled
  contributors: predictive optimization enabled
  electoral_events: predictive optimization enabled
  contributions: predictive optimization enabled

PHASE 2 COMPLETE — Silver Layer Summary
  pfin_dev.silver.recipients: 1,004 rows
  pfin_dev.silver.contributors: 123,453 rows
  pfin_dev.silver.electoral_events: 44 rows
  pfin_dev.silver.contributions: 282,098 rows
  Timestamp: 2026-06-01 02:35:41.678293


In [0]:
# Encoding fix check
spark.sql("SELECT DISTINCT political_party FROM pfin_dev.silver.recipients ORDER BY political_party").show(20, truncate=False)

# Province check
spark.sql("SELECT contributor_province, COUNT(*) as cnt FROM pfin_dev.silver.contributors GROUP BY contributor_province ORDER BY cnt DESC").show(20)

+----------------------------------+
|political_party                   |
+----------------------------------+
|Bloc QuÃÂ©bÃÂ©cois              |
|Canadian Future Party             |
|Centrist Party of Canada          |
|Christian Heritage Party of Canada|
|Communist Party of Canada         |
|Conservative Party of Canada      |
|Green Party of Canada             |
|Independent                       |
|Liberal Party of Canada           |
|Libertarian Party of Canada       |
|Marxist-Leninist Party of Canada  |
|New Democratic Party              |
|No Affiliation                    |
|Parti RhinocÃÂ©ros Party         |
|People's Party of Canada          |
|United Party of Canada (UP)       |
+----------------------------------+

+--------------------+-----+
|contributor_province|  cnt|
+--------------------+-----+
|                  ON|53057|
|                  BC|23379|
|                  AB|20296|
|                  QC| 9670|
|                  MB| 4212|
|                  SK| 4131

In [0]:
from pyspark.sql import functions as F

table = "pfin_dev.silver.recipients"
df = spark.table(table)

# Show the actual broken values to confirm
df.filter(F.col("political_party").contains("Â")).select("political_party").distinct().show(truncate=False)

# Fix by exact match
df_fixed = df.withColumn("political_party",
    F.when(F.col("political_party").contains("QuÃÂ©bÃÂ©cois"), "Bloc Québécois")
     .when(F.col("political_party").contains("RhinocÃÂ©ros"), "Parti Rhinocéros Party")
     .otherwise(F.col("political_party"))
)

df_fixed.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)

# Verify
spark.sql(f"SELECT DISTINCT political_party FROM {table} ORDER BY political_party").show(20, truncate=False)

+-------------------------+
|political_party          |
+-------------------------+
|Bloc QuÃÂ©bÃÂ©cois     |
|Parti RhinocÃÂ©ros Party|
+-------------------------+

+----------------------------------+
|political_party                   |
+----------------------------------+
|Bloc QuÃÂ©bÃÂ©cois              |
|Canadian Future Party             |
|Centrist Party of Canada          |
|Christian Heritage Party of Canada|
|Communist Party of Canada         |
|Conservative Party of Canada      |
|Green Party of Canada             |
|Independent                       |
|Liberal Party of Canada           |
|Libertarian Party of Canada       |
|Marxist-Leninist Party of Canada  |
|New Democratic Party              |
|No Affiliation                    |
|Parti RhinocÃÂ©ros Party         |
|People's Party of Canada          |
|United Party of Canada (UP)       |
+----------------------------------+



In [0]:
from pyspark.sql import functions as F

table = "pfin_dev.silver.recipients"
df = spark.table(table)

df_fixed = df.withColumn("political_party",
    F.when(
        F.col("political_party").contains("Bloc") & F.col("political_party").contains("cois"),
        F.lit("Bloc Québécois")
    ).when(
        F.col("political_party").contains("Rhinoc") & F.col("political_party").contains("Party"),
        F.lit("Parti Rhinocéros Party")
    ).otherwise(F.col("political_party"))
)

df_fixed.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)

# Verify
spark.sql(f"SELECT DISTINCT political_party FROM {table} ORDER BY political_party").show(20, truncate=False)

+----------------------------------+
|political_party                   |
+----------------------------------+
|Bloc Québécois                    |
|Canadian Future Party             |
|Centrist Party of Canada          |
|Christian Heritage Party of Canada|
|Communist Party of Canada         |
|Conservative Party of Canada      |
|Green Party of Canada             |
|Independent                       |
|Liberal Party of Canada           |
|Libertarian Party of Canada       |
|Marxist-Leninist Party of Canada  |
|New Democratic Party              |
|No Affiliation                    |
|Parti Rhinocéros Party            |
|People's Party of Canada          |
|United Party of Canada (UP)       |
+----------------------------------+



In [0]:
spark.sql("SELECT DISTINCT electoral_event FROM pfin_dev.silver.electoral_events").show(truncate=False)
spark.sql("SELECT DISTINCT electoral_district FROM pfin_dev.silver.recipients WHERE electoral_district LIKE '%Â%'").show(truncate=False)
spark.sql("SELECT DISTINCT recipient_name FROM pfin_dev.silver.recipients WHERE recipient_name LIKE '%Â%'").show(truncate=False)

+-----------------------------+
|electoral_event              |
+-----------------------------+
|NULL                         |
|Quarterly                    |
|August 18, 2025, By-election |
|Annual                       |
|45th general election        |
|April 14th, 2025, By-election|
+-----------------------------+

+------------------+
|electoral_district|
+------------------+
+------------------+

+--------------+
|recipient_name|
+--------------+
+--------------+



In [0]:
from pyspark.sql import functions as F

# ── Helper: fix double-encoded UTF-8 ────────────────────────
def fix_encoding_col(df, col_name):
    return df.withColumn(col_name,
        F.when(
            F.col(col_name).contains("Â"),
            F.decode(F.encode(F.col(col_name), "ISO-8859-1"), "UTF-8")
        ).otherwise(F.col(col_name))
    )

# ── 1. Fix recipients ───────────────────────────────────────
print("Fixing recipients...")
df_r = spark.table("pfin_dev.silver.recipients")
for col in ["recipient_name", "recipient_last_name", "electoral_district"]:
    df_r = fix_encoding_col(df_r, col)
df_r.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("pfin_dev.silver.recipients")

# ── 2. Fix contributors (French names/cities) ───────────────
print("Fixing contributors...")
df_c = spark.table("pfin_dev.silver.contributors")
for col in ["contributor_name", "contributor_last_name", "contributor_first_name", "contributor_city"]:
    df_c = fix_encoding_col(df_c, col)
df_c.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("pfin_dev.silver.contributors")

# ── 3. Fix contributions (leadership_contestant) ────────────
print("Fixing contributions...")
df_x = spark.table("pfin_dev.silver.contributions")
df_x = fix_encoding_col(df_x, "leadership_contestant")
df_x.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("pfin_dev.silver.contributions")

# ── Verify ───────────────────────────────────────────────────
print("\n=== Verification ===")
spark.sql("SELECT DISTINCT electoral_district FROM pfin_dev.silver.recipients WHERE electoral_district LIKE '%Â%'").show()
spark.sql("SELECT DISTINCT recipient_name FROM pfin_dev.silver.recipients WHERE recipient_name LIKE '%Â%'").show()
spark.sql("SELECT DISTINCT contributor_city FROM pfin_dev.silver.contributors WHERE contributor_city LIKE '%Â%'").show()
print("If all three queries return empty, encoding is clean.")

Fixing recipients...
Fixing contributors...
Fixing contributions...

=== Verification ===
+------------------+
|electoral_district|
+------------------+
+------------------+

+--------------+
|recipient_name|
+--------------+
+--------------+

+----------------+
|contributor_city|
+----------------+
+----------------+

If all three queries return empty, encoding is clean.


In [0]:
from pyspark.sql import functions as F

df_c = spark.table("pfin_dev.silver.contributors")

# Fix MONTRÃÂ©AL → MONTRÉAL (needs double decode pass)
df_c = df_c.withColumn("contributor_city",
    F.when(
        F.col("contributor_city").contains("Â"),
        F.decode(F.encode(
            F.decode(F.encode(F.col("contributor_city"), "ISO-8859-1"), "UTF-8"),
        "ISO-8859-1"), "UTF-8")
    ).otherwise(F.col("contributor_city"))
)

# Strip any remaining  characters (garbage like DELEAUÂ)
df_c = df_c.withColumn("contributor_city",
    F.regexp_replace(F.col("contributor_city"), "Â", "")
)
df_c = df_c.withColumn("contributor_city", F.trim(F.col("contributor_city")))

df_c.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("pfin_dev.silver.contributors")

# Verify
spark.sql("SELECT DISTINCT contributor_city FROM pfin_dev.silver.contributors WHERE contributor_city LIKE '%Â%'").show()

+----------------+
|contributor_city|
+----------------+
+----------------+



In [0]:
from pyspark.sql import functions as F

# Check ALL text columns in ALL silver tables for encoding artifacts
markers = ["Â", "Ã¯Â»Â¿", "Ã©", "Ã¨", "Ã´", "Ãª", "Ã§"]

for table_name in ["recipients", "contributors", "electoral_events", "contributions"]:
    full_name = f"pfin_dev.silver.{table_name}"
    df = spark.table(full_name)
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]
    
    print(f"\n=== {full_name} ===")
    for col in text_cols:
        for marker in markers:
            cnt = df.filter(F.col(col).contains(marker)).count()
            if cnt > 0:
                print(f"  {col}: {cnt} rows contain '{marker}'")
                df.filter(F.col(col).contains(marker)).select(col).distinct().show(5, truncate=False)


=== pfin_dev.silver.recipients ===
  recipient_name: 38 rows contain 'Ã©'
+------------------------------+
|recipient_name                |
+------------------------------+
|FrÃ©chette, Mario             |
|de Vries, RenÃ©               |
|GÃ©nÃ©reux, Bernard           |
|FrÃ©chette, FranÃ§ois         |
|Bloc QuÃ©bÃ©cois de Saint-Jean|
+------------------------------+
only showing top 5 rows
  recipient_name: 4 rows contain 'Ã¨'
+-------------------+
|recipient_name     |
+-------------------+
|BriÃ¨re, Ãlisabeth|
|MendÃ¨s, Alexandra |
|Duplessis, EugÃ¨ne |
|Gill, MarilÃ¨ne    |
+-------------------+

  recipient_name: 2 rows contain 'Ãª'
+--------------------------------+
|recipient_name                  |
+--------------------------------+
|DeschÃªnes, GaÃ«tan             |
|DeschÃªnes-ThÃ©riault, Guillaume|
+--------------------------------+

  recipient_name: 2 rows contain 'Ã§'
+---------------------+
|recipient_name       |
+---------------------+
|FrÃ©chette, FranÃ§ois|
|Morin

In [0]:
from pyspark.sql import functions as F

markers_check = ["Â", "Ã©", "Ã¨", "Ã´", "Ãª", "Ã§"]

for table_name in ["recipients", "contributors", "electoral_events", "contributions"]:
    full_name = f"pfin_dev.silver.{table_name}"
    print(f"Fixing {full_name}...")
    df = spark.table(full_name)
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]

    for col_name in text_cols:
        c = F.col(col_name)
        df = df.withColumn(col_name, F.when(c.contains("Ã"), F.decode(F.encode(c, "ISO-8859-1"), "UTF-8")).otherwise(c))
        c2 = F.col(col_name)
        df = df.withColumn(col_name, F.when(c2.contains("Ã"), F.decode(F.encode(c2, "ISO-8859-1"), "UTF-8")).otherwise(c2))
        df = df.withColumn(col_name, F.regexp_replace(F.col(col_name), "Â", ""))
        df = df.withColumn(col_name, F.trim(F.col(col_name)))

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)

print("\n=== Final Verification ===")
clean = True
for table_name in ["recipients", "contributors", "electoral_events", "contributions"]:
    full_name = f"pfin_dev.silver.{table_name}"
    df = spark.table(full_name)
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]
    for col_name in text_cols:
        for marker in markers_check:
            cnt = df.filter(F.col(col_name).contains(marker)).count()
            if cnt > 0:
                print(f"  STILL BROKEN: {full_name}.{col_name} — {cnt} rows contain '{marker}'")
                clean = False

if clean:
    print("  ALL CLEAN — no encoding artifacts remaining.")

print("\n=== Spot Check ===")
spark.sql("SELECT DISTINCT political_party FROM pfin_dev.silver.recipients WHERE political_party LIKE '%Bloc%' OR political_party LIKE '%Parti%'").show(truncate=False)
spark.sql("SELECT DISTINCT electoral_district FROM pfin_dev.silver.recipients WHERE electoral_district LIKE '%bec%' OR electoral_district LIKE '%ivi%' OR electoral_district LIKE '%te-%'").show(truncate=False)
spark.sql("SELECT contributor_city, COUNT(*) as cnt FROM pfin_dev.silver.contributors WHERE contributor_city LIKE '%MONTR%' GROUP BY contributor_city").show(truncate=False)

Fixing pfin_dev.silver.recipients...
Fixing pfin_dev.silver.contributors...
Fixing pfin_dev.silver.electoral_events...
Fixing pfin_dev.silver.contributions...

=== Final Verification ===
  ALL CLEAN — no encoding artifacts remaining.

=== Spot Check ===
+----------------------+
|political_party       |
+----------------------+
|Bloc Québécois        |
|Parti Rhinocéros Party|
+----------------------+

+--------------------------------------------------+
|electoral_district                                |
+--------------------------------------------------+
|Côte-du-Sud-Rivière-du-Loup-Kataskomiq-Témiscouata|
|Rosemont--La Petite-Patrie                        |
|Saint John--Kennebecasis                          |
|Laurier--Sainte-Marie                             |
|Rivière-des-Mille-Îles                            |
|La Pointe-de-l'Île                                |
|Charlesbourg--Haute-Saint-Charles                 |
|Québec Centre                                     |
|Côte-Nord--

In [0]:
from pyspark.sql import functions as F

df = spark.table("pfin_dev.silver.contributors")

# 1. Re-apply UPPER to normalize é → É (accent case mismatch)
df = df.withColumn("contributor_city", F.upper(F.col("contributor_city")))

# 2. Fix the last 2 garbled variants (20 rows total)
df = df.withColumn("contributor_city",
    F.when(F.col("contributor_city").startswith("MONTR") & F.col("contributor_city").contains("AL") 
           & ~F.col("contributor_city").isin("MONTREAL", "MONTRÉAL", "MONTRÉAL-NORD", "MONTRÉAL-OUEST", 
                                              "MONTREAL-NORD", "MONTREAL-OUEST", "MONTREAL OUEST", 
                                              "MONTREAL WEST", "MONTREAL-LACHINE", "MONTREAL. ARR ST LAURENT",
                                              "LASALLE MONTREAL", "WEST MONTROSE", "MONTROSE"),
        F.regexp_replace(F.col("contributor_city"), "MONTR.*?AL", "MONTRÉAL")
    ).otherwise(F.col("contributor_city"))
)

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("pfin_dev.silver.contributors")

# Verify
spark.sql("SELECT contributor_city, COUNT(*) as cnt FROM pfin_dev.silver.contributors WHERE contributor_city LIKE '%MONTR%' GROUP BY contributor_city ORDER BY cnt DESC").show(30, truncate=False)

+------------------------+----+
|contributor_city        |cnt |
+------------------------+----+
|MONTRÉAL                |1363|
|MONTREAL                |882 |
|MONTRÉAL-OUEST          |25  |
|MONTRÉAL-NORD           |17  |
|WEST MONTROSE           |14  |
|MONTREAL WEST           |11  |
|MONTREAL-OUEST          |6   |
|MONTREAL-NORD           |6   |
|MONTREAL OUEST          |2   |
|MONTREAL-LACHINE        |1   |
|MONTROSE                |1   |
|MONTRÉAL -NORD          |1   |
|LASALLE MONTREAL        |1   |
|MONTREAL. ARR ST LAURENT|1   |
|MONTRÉALL               |1   |
+------------------------+----+

